In [1]:
# --- GPU + Mixed Precision Setup (start cell) ---
import os
import tensorflow as tf
from tensorflow.keras import mixed_precision # Some calculations are in float16 instead of float32, it helps to train model much faster and uses less VRAM

# Hide unnecessary info
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Check if GPU is loaded
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU detected: {gpus[0].name}")
else:
    print("GPU NOT detected — TensorFlow will run on CPU")


mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision enabled (mixed_float16)")

print(f"TensorFlow version: {tf.__version__}")
print(f"Compute dtype: {mixed_precision.global_policy().compute_dtype}")
print(f"Variable dtype: {mixed_precision.global_policy().variable_dtype}")


2025-12-10 17:20:33.678191: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU detected: /physical_device:GPU:0
Mixed precision enabled (mixed_float16)
TensorFlow version: 2.20.0
Compute dtype: float16
Variable dtype: float32


In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.image import resize
import matplotlib.pyplot as plt

import librosa

data_folder = "../Data/genres_original"
classes = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock']
def preprocessing_wav_to_melspectogram(data_folder, classes, target_shape = (150,150), chunk_duration = 4, overlap_duration = 2):
    melspectograms_list = []
    labels_list = []
    for class_number, class_name in enumerate(classes):
        class_folder = os.path.join(data_folder, class_name)
        print(f'Processing of class {class_name} data is ongoing')
        for filename in os.listdir(class_folder):
            if filename.endswith('.wav'):
                file_path = os.path.join(class_folder, filename)
                audio_data, sample_rate = librosa.load(file_path, sr = None)
                chunk_samples = int(sample_rate * chunk_duration)
                overlap_samples = int(sample_rate * overlap_duration)

                num_of_chunks = int(np.ceil((len(audio_data) - chunk_samples) /(chunk_samples -  overlap_samples)))+1

                for i in range(num_of_chunks):
                    start = i * (chunk_samples - overlap_samples)
                    end = start + chunk_samples

                    chunk = audio_data[start:end]

                    mel_spectogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate)

                    mel_spectogram = resize(np.expand_dims(mel_spectogram, axis = -1),target_shape)

                    melspectograms_list.append(mel_spectogram)
                    labels_list.append(class_number)
    return np.array(melspectograms_list), np.array(labels_list)
data, labels = preprocessing_wav_to_melspectogram(data_folder, classes)
from tensorflow.keras.utils import to_categorical
labels = to_categorical(labels,num_classes = len(classes))
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(data,labels,test_size=0.2,random_state=42)

Processing of class blues data is ongoing


I0000 00:00:1763657771.653579     946 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3539 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Processing of class classical data is ongoing
Processing of class country data is ongoing
Processing of class disco data is ongoing
Processing of class hiphop data is ongoing
Processing of class jazz data is ongoing
Processing of class metal data is ongoing
Processing of class pop data is ongoing
Processing of class reggae data is ongoing
Processing of class rock data is ongoing


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, Flatten, Dense, Dropout, Input

# Zdefiniuj kluczowe stałe
INPUT_SHAPE = (150, 150, 1) # Wymiar Twoich spektrogramów (Wysokość, Szerokość, Kanały)
NUM_CLASSES = 10           # Liczba gatunków muzycznych

# Użyj modelu Sequential
second_model = Sequential([
    # Warstwa wejściowa (Input Layer)
    Input(shape=INPUT_SHAPE),

    # Blok 1: Conv1 + Pool1 + BatchNorm1
    # Conv1: 32 filtry, rozmiar jądra 3x3, padding='same'
    Conv2D(filters=32, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu', name='conv1'),
    # BatchNorm1: Normalizacja wsadowa (PyTorch 'bn1')
    BatchNormalization(name='bn1'),
    # Pool1: Redukcja wymiaru 2x2, stride=2
    MaxPool2D(pool_size=(2, 2), strides=(2, 2), name='pool1'),

    # Blok 2: Conv2 + Pool2 + BatchNorm2
    # Conv2: 64 filtry, rozmiar jądra 3x3, padding='same'
    Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu', name='conv2'),
    # BatchNorm2: Normalizacja wsadowa (PyTorch 'bn2')
    BatchNormalization(name='bn2'),
    # Pool2: Redukcja wymiaru 2x2, stride=2
    MaxPool2D(pool_size=(2, 2), strides=(2, 2), name='pool2'),

    # Przekształcanie do wektora dla warstw gęstych
    Flatten(name='flatten'),

    # Warstwy Gęste (Fully Connected)
    # FC1: 128 jednostek (PyTorch 'fc1')
    Dense(units=128, activation='relu', name='fc1'),
    # Dropout: Prawdopodobieństwo 0.5 (PyTorch 'dropout1')
    Dropout(rate=0.5, name='dropout1'),

    # FC2 (Wyjściowa): 10 jednostek, 'softmax' do klasyfikacji (PyTorch 'fc2')
    Dense(units=NUM_CLASSES, activation='softmax', name='fc2')
])

# Wyświetl podsumowanie modelu
second_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 150, 150, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 150, 150, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 75, 75, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 87616)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 128)            │    11,214,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,235,466 (42.86 MB)

 Trainable params: 11,235,274 (42.86 MB)

 Non-trainable params: 192 (768.00 B)

In [4]:
from tensorflow.keras.optimizers import Adam

# Kompilacja modelu
second_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model został zdefiniowany i skompilowany, gotowy do treningu.")

Model został zdefiniowany i skompilowany, gotowy do treningu.


In [5]:
if 'X_train' in locals() and 'Y_train' in locals():
    print("\nRozpoczynanie treningu (30 epok, batch_size=32)...")
    training_history = second_model.fit(
        X_train,
        Y_train,
        epochs=30,
        batch_size=32,
        validation_data=(X_test, Y_test)
    )
    print("\nTrening zakończony!")
else:
    print("\n--- UWAGA: Dane treningowe (X_train, Y_train) nie są dostępne. ---")
    print("Nie można uruchomić model.fit. Upewnij się, że wykonałeś preprocessing i podział danych (komórki 1-8).")


Rozpoczynanie treningu (30 epok, batch_size=32)...


2025-11-20 17:59:30.627691: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1078200000 exceeds 10% of free system memory.
2025-11-20 17:59:34.947936: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1078200000 exceeds 10% of free system memory.


Epoch 1/30


2025-11-20 17:59:39.332467: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f0770006240 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-20 17:59:39.332514: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-11-20 17:59:40.615108: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-20 17:59:42.731166: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91600
2025-11-20 17:59:43.505742: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


  3/375 ━━━━━━━━━━━━━━━━━━━━ 16s 44ms/step - accuracy: 0.1684 - loss: 3.8485   

I0000 00:00:1763658010.202419    2461 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


375/375 ━━━━━━━━━━━━━━━━━━━━ 60s 70ms/step - accuracy: 0.2641 - loss: 2.2212 - val_accuracy: 0.3573 - val_loss: 1.8557
Epoch 2/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3568 - loss: 1.8428 - val_accuracy: 0.4461 - val_loss: 1.6053
Epoch 3/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.4199 - loss: 1.6587 - val_accuracy: 0.5516 - val_loss: 1.3988
Epoch 4/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.4817 - loss: 1.4858 - val_accuracy: 0.5860 - val_loss: 1.3007
Epoch 5/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.5277 - loss: 1.3475 - val_accuracy: 0.6287 - val_loss: 1.1992
Epoch 6/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5602 - loss: 1.2319 - val_accuracy: 0.6361 - val_loss: 1.1640
Epoch 7/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5928 - loss: 1.1281 - val_accuracy: 0.6561 - val_loss: 1.1239
Epoch 8/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.6295 - loss: 1.0258 - val_accuracy: 0.69